In [2]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

# -------------------------------------------------------------
# 0. Global Dark Style Configuration
# -------------------------------------------------------------
plt.style.use('dark_background')

# -------------------------------------------------------------
# 1. Physics & Kinematics Parameters (Exact One Cycloid Period)
# -------------------------------------------------------------
R = 1.0                           # Cyclotron radius (Larmor radius)
omega = 1.5                       # Cyclotron frequency (rad/s)
v0 = omega * R                    # Forward speed of the box (1.5 m/s)
T_period = 2.0 * np.pi / omega    # Exact period for one full loop (~4.19 s)

n_frames = 120
t = np.linspace(0, T_period, n_frames)

# Lab Frame (S): Particle in uniform circular motion
x_lab = R * np.sin(omega * t)
y_lab = R * (1.0 - np.cos(omega * t))

# Moving Box in Lab Frame
box_w = 2.0
box_h = 2.4
x_box = v0 * t - box_w * 0.4
y_box = -0.2

# Box Frame (S'): Particle traces the Cycloid
x_prime = R * (np.sin(omega * t) - omega * t)
y_prime = R * (1.0 - np.cos(omega * t))

# -------------------------------------------------------------
# 2. Dual-Panel Figure Setup (Pure Black Background)
# -------------------------------------------------------------
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5.0), dpi=100, facecolor='#000000')

for ax in (ax1, ax2):
    ax.set_facecolor('#000000')
    ax.grid(True, linestyle='--', color='#1e293b', alpha=0.6)
    for spine in ax.spines.values():
        spine.set_color('#334155')
    ax.tick_params(colors='#94a3b8', labelsize=8.5)

# Frame S (Lab) Axes
ax1.set_xlim(-1.5, 7.5)
ax1.set_ylim(-0.6, 2.7)
ax1.set_aspect('equal')
ax1.set_title("Frame S (Lab): Pure Magnetic Deflection", fontsize=11, pad=12, color='#ffffff', weight='bold')
ax1.set_xlabel("x (Laboratory Frame)", fontsize=9, color='#94a3b8')
ax1.set_ylabel("y", fontsize=9, color='#94a3b8')

# Frame S' (Box) Axes
ax2.set_xlim(-7.5, 1.5)
ax2.set_ylim(-0.6, 2.7)
ax2.set_aspect('equal')
ax2.set_title("Frame S' (Box): Dual Fields & Cycloid Path", fontsize=11, pad=12, color='#ffffff', weight='bold')
ax2.set_xlabel("x' (Relative to Box Origin)", fontsize=9, color='#94a3b8')
ax2.set_ylabel("y'", fontsize=9, color='#94a3b8')

# -------------------------------------------------------------
# 3. Static Field Overlays (Luminous Contrast)
# -------------------------------------------------------------
# Panel 1: Uniform Magnetic Field B (into page)
gx1, gy1 = np.meshgrid(np.linspace(-1.0, 7.0, 9), np.linspace(0.0, 2.4, 4))
ax1.scatter(gx1, gy1, marker='o', s=80, facecolors='none', edgecolors='#38bdf8', alpha=0.35, lw=1.2)
ax1.scatter(gx1, gy1, marker='x', s=35, color='#38bdf8', alpha=0.35, lw=1.2)
ax1.text(7.2, 2.45, r"$\mathbf{B} = -B_0\hat{\mathbf{z}}$ ($\otimes$)", color='#38bdf8', fontsize=9, ha='right', weight='bold')

# Panel 2: Induced Electric Field E' (upward arrows) & B' Field
gx2, gy2 = np.meshgrid(np.linspace(-7.0, 1.0, 9), np.linspace(0.0, 2.4, 4))
ax2.scatter(gx2, gy2, marker='o', s=80, facecolors='none', edgecolors='#38bdf8', alpha=0.35, lw=1.2)
ax2.scatter(gx2, gy2, marker='x', s=35, color='#38bdf8', alpha=0.35, lw=1.2)

# Electric Field vectors pointing vertically upwards
for ex in np.linspace(-6.5, 0.5, 8):
    ax2.annotate('', xy=(ex, 2.2), xytext=(ex, 0.2),
                 arrowprops=dict(arrowstyle="->", color='#facc15', lw=1.3, alpha=0.45))

ax2.text(1.2, 2.45, r"$\mathbf{E}' = +v_0 B_0\hat{\mathbf{y}}'$", color='#facc15', fontsize=9, ha='right', weight='bold')
ax2.text(-7.2, 2.45, r"$\mathbf{B}' = -B_0\hat{\mathbf{z}}'$", color='#38bdf8', fontsize=9, ha='left', weight='bold')

# Floor and Compartment Outline in Frame S'
ax2.axhline(0.0, color='#475569', lw=1.5, linestyle='-', alpha=0.8)
box_interior = Rectangle((-2.0 * np.pi * R - 0.4, 0.0), 2.0 * np.pi * R + 1.2, 2.3,
                         edgecolor='#f59e0b', facecolor='#f59e0b', alpha=0.08, lw=1.8, linestyle='--')
ax2.add_patch(box_interior)

# -------------------------------------------------------------
# 4. Animated Elements
# -------------------------------------------------------------
# Panel 1 (Lab): Moving Box, Observer, Particle, and Circle Trail
box_patch = Rectangle((x_box[0], y_box), box_w, box_h,
                      edgecolor='#f59e0b', facecolor='#f59e0b', alpha=0.15, lw=2.0)
ax1.add_patch(box_patch)

obs_dot_1, = ax1.plot([], [], 'o', color='#00f0ff', markersize=7)
trail_1, = ax1.plot([], [], color='#f43f5e', lw=2.0, alpha=0.6)
particle_1, = ax1.plot([], [], 'o', color='#ff3366', markersize=8, markeredgecolor='#ffffff', markeredgewidth=1.0)

# Panel 2 (Box Frame): Stationary Observer, Cycloid Trail, and Particle
obs_dot_2, = ax2.plot([0.35], [0.35], 'o', color='#00f0ff', markersize=7)
ax2.text(0.35, 0.55, 'Observer (At Rest)', fontsize=7.5, color='#00f0ff', ha='center', weight='bold')

trail_2, = ax2.plot([], [], color='#f43f5e', lw=2.2, alpha=0.85)
particle_2, = ax2.plot([], [], 'o', color='#ff3366', markersize=8, markeredgecolor='#ffffff', markeredgewidth=1.0)

# Formula badges
ax1.text(-1.2, -0.45, r"$\mathbf{F} = q(\mathbf{v} \times \mathbf{B})$ (Circular Arc)", fontsize=8, color='#64748b')
ax2.text(-7.2, -0.45, r"$\mathbf{F}' = q\mathbf{E}' + q(\mathbf{v}' \times \mathbf{B}')$ (Cycloid)", fontsize=8, color='#64748b')

# -------------------------------------------------------------
# 5. Animation Update Function
# -------------------------------------------------------------
def update(i):
    # 1. Update Lab Frame elements
    box_patch.set_x(x_box[i])
    obs_x = x_box[i] + 0.35
    obs_y = y_box + 0.5
    obs_dot_1.set_data([obs_x], [obs_y])
    
    trail_1.set_data(x_lab[:i+1], y_lab[:i+1])
    particle_1.set_data([x_lab[i]], [y_lab[i]])
    
    # 2. Update Box Frame elements
    trail_2.set_data(x_prime[:i+1], y_prime[:i+1])
    particle_2.set_data([x_prime[i]], [y_prime[i]])
    
    return box_patch, obs_dot_1, trail_1, particle_1, trail_2, particle_2

# -------------------------------------------------------------
# 6. Render, Save & Display (Enforcing Black Frame Export)
# -------------------------------------------------------------
anim = FuncAnimation(fig, update, frames=n_frames, interval=50, blit=True)

# Save with exact black background padding
anim.save('cycloid_invariance.mp4', writer='ffmpeg', fps=20,
          savefig_kwargs={'facecolor': '#000000'},
          extra_args=['-vcodec', 'libx264'])
plt.close(fig)

# Display interactive player in Kaggle
HTML(anim.to_html5_video())